# HACA — Domain-adaptation v3 (Kaggle T4)

Trains the in-domain sentiment model on **HACA-Sent v3** (cleaned + Claude-labelled SRT pool
+ synthetic) and benchmarks it against a rubric-prompted Atlas-Chat LLM, all on the frozen
gold test `domaine_reel_v2`.

**How to use**
1. New Notebook → Settings → Accelerator: **GPU T4 x1**, Internet: **On**.
2. Set `GITHUB_REPO` in cell 2 to your repo URL (branch `feat/haca-sent-v3`).
3. Run cells top to bottom. Encoder fine-tune ≈ 30 min; LLM eval ≈ 10 min.

**Goal:** macro-F1 ≥ 0.70 on `domaine_reel_v2`, pos-F1 ≥ ~0.45 (was 0.19), no darija_ar regression.

In [ ]:
# 1 — Install deps not in Kaggle's default image
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable,'-m','pip','install','-q',*a])
pip('bitsandbytes>=0.43.0','pysrt>=1.1.2','seaborn>=0.13.0')
print('deps ok')

In [ ]:
# 2 — Clone the repo (use the branch that has HACA-Sent v3)
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/haca-benchmark.git'  # <-- CHANGE ME
BRANCH      = 'feat/haca-sent-v3'

import os, sys, subprocess
WORK='/kaggle/working'; REPO=os.path.join(WORK,'benchmark')
if not os.path.isdir(REPO):
    subprocess.check_call(['git','clone','-b',BRANCH,GITHUB_REPO,REPO])
else:
    subprocess.check_call(['git','-C',REPO,'pull'])
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO,'src'))
print('repo ready at', REPO)

In [ ]:
# 3 — Download MAC raw corpus (marbertv2-haca = MAC + HACA-Sent mixed training)
import requests, os
RAW=os.path.join(REPO,'data','raw'); os.makedirs(os.path.join(RAW,'MAC'),exist_ok=True)
dest=os.path.join(RAW,'MAC','MAC corpus.csv')
if not os.path.exists(dest):
    url='https://raw.githubusercontent.com/LeMGarouani/MAC/master/MAC%20corpus.csv'
    r=requests.get(url,timeout=60); r.raise_for_status(); open(dest,'wb').write(r.content)
    print('MAC downloaded', len(r.content)//1024,'KB')
else:
    print('MAC already present')

In [ ]:
# 4 — GPU check + (re)build & verify the HACA-Sent v3 training set
import torch, pandas as pd
from src.utils import set_seeds; set_seeds()
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# haca_train_v3.csv is committed; this just reassembles + sanity-checks it.
subprocess.check_call([sys.executable,'src/build_haca_train.py'])
df=pd.read_csv('data/test_sets/haca_train_v3.csv')
print('train rows:', len(df), '| dist:', dict(df.label.value_counts()))
gold=pd.read_csv('data/test_sets/domaine_reel_v2.csv')
assert len(set(df.text) & set(gold.text))==0, 'LEAK: train overlaps gold test!'
print('gold test:', len(gold), '| no train/test text overlap ✓')

## Stage 5 — Fine-tune the in-domain encoder
`marbertv2-haca`: MARBERTv2 on MAC + HACA-Sent v3, class-weighted **focal loss** + ×3 pos
oversampling. Writes `results/marbertv2-haca_domaine_reel_v2.json` and
`results/marbertv2-haca_darija_ar.json`.

In [ ]:
# 5 — Fine-tune marbertv2-haca  (~30 min on T4)
import time
from src.finetune import finetune
t0=time.time(); finetune('marbertv2-haca'); print(f'\nwall: {(time.time()-t0)/60:.1f} min')

In [ ]:
# 6 — Threshold calibration on the gold test (near-free gain)
subprocess.check_call([sys.executable,'src/calibrate_thresholds.py','--model','marbertv2-haca'])

In [ ]:
# 7 — Show results vs the 0.70 target
import json, os
for f in sorted(os.listdir('results')):
    if f.startswith('marbertv2-haca') and f.endswith('.json'):
        d=json.load(open('results/'+f)); rep=d.get('classification_report',{})
        print(f"\n{f}\n  macro-F1: {d['macro_f1']}")
        for c in ['neg','neu','pos']:
            if c in rep: print(f"  {c}: F1={rep[c]['f1-score']:.3f}  (n={int(rep[c]['support'])})")

## Stage 6 — Rubric-prompted LLM baseline
Atlas-Chat-2B *told* the content-valence rubric (+ few-shot), on the same gold test. The
content-valence task is rubric-following, so an instructable LLM may beat the encoder here.

In [ ]:
# 8 — Rubric-prompted Atlas-Chat-2B  (~10 min on T4, 4-bit)
subprocess.check_call([sys.executable,'src/eval_llm_rubric.py','--model','2b'])
import json
d=json.load(open('results/atlas-chat-2b-rubric_domaine_reel_v2.json'))
print('Atlas-Chat-2B (rubric) macro-F1:', d['macro_f1'])

## Optional — permissive-licence encoder for HACA production
MARBERTv2 is **research-only**. `darijabert-haca` (DarijaBERT base) is the deployable fallback.

In [ ]:
# 9 — (optional) darijabert-haca
from src.finetune import finetune
finetune('darijabert-haca')
subprocess.check_call([sys.executable,'src/calibrate_thresholds.py','--model','darijabert-haca'])

In [ ]:
# 10 — Package checkpoints + results for download (Output tab)
import shutil, os
for key in ['marbertv2-haca','darijabert-haca']:
    p=f'checkpoints/{key}'
    if os.path.isdir(p):
        shutil.make_archive(f'/kaggle/working/{key}','zip',p); print('zipped', key)
shutil.make_archive('/kaggle/working/results','zip','results'); print('zipped results')